## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [18]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
import requests
import json
from mlflow.models import infer_signature
...

Ellipsis

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [3]:
# Implement transformation logic as in session 2

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

categorical_features = ['Geography', 'Gender']

numerical_features = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'HasCrCard',
    'IsActiveMember',
    'EstimatedSalary'
]

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['CreditScore', 'Age', 'Tenure', 'Balance',
                                  'NumOfProducts', 'HasCrCard',
                                  'IsActiveMember', 'EstimatedSalary']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Geography', 'Gender'])])

In [5]:
import joblib

# fit encoder on dataset

df = pd.read_csv('Churn_Modelling_train_test.csv')

X = df.drop(columns=['Exited', 'RowNumber', 'CustomerId', 'Surname'])

preprocessor.fit(X)

# get categorical encoder

encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']

# save encoder

PATH = "./"

joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

print("Encoder saved successfully!")

Encoder saved successfully!


In [6]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [7]:
# import validation dataset to test inference

df_validation = pd.read_csv('Churn_Modelling_val.csv')

df_validation.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90,0
1,4685,15736963,Herring,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30,0
2,1732,15721730,Amechi,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31,0
3,4743,15762134,Liang,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74,0
4,4522,15648898,Chuang,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79,0


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [8]:
# transform data - if necessary

# remove columns not used during training

df_validation_transformed = df_validation.drop(
    columns=['Exited', 'RowNumber', 'CustomerId', 'Surname'],
    errors='ignore'
)

df_validation_transformed.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90
1,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30
2,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31
3,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74
4,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79


##### Batch Inference

In [9]:
# define a function to implement batch inference with mlflow

def batch_inference(model_uri: str, input: pd.DataFrame):

    # load model
    model = mlflow.pyfunc.load_model(model_uri)

    # make predictions
    predictions = model.predict(input)

    return predictions

In [10]:
# define the model uri that should be used

model_uri = "runs:/50cedc1a0d754078b75019a45fa58d30/decision_tree_model"

batch_prediction_result = batch_inference(
    model_uri,
    df_validation_transformed
)

batch_prediction_result

array([0, 0, 0, ..., 0, 0, 0])

In [11]:
# check the confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df_validation['Exited'],
    batch_prediction_result
)

print(cm)

[[777  26]
 [116  82]]


##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [19]:
import requests
import json

In [12]:
# import validation dataset to test inference - just one record

df_validation = pd.read_csv('Churn_Modelling_val.csv').head(1)

df_validation

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.0,2,1.0,1.0,74900.9,0


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [13]:
# transform data - if necessary

df_validation_transformed = df_validation.drop(
    columns=['Exited', 'RowNumber', 'CustomerId', 'Surname'],
    errors='ignore'
)

df_validation_transformed

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,807,Spain,Male,42.0,5,0.0,2,1.0,1.0,74900.9


In [14]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

In [15]:
# define a function to implement online inference with mlflow - pandas input

def online_inference_pandas(url: str, input: pd.DataFrame):

    data = {
        "dataframe_split": input.to_dict(orient="split")
    }

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        data=json.dumps(data)
    )

    return response

In [20]:
response_pandas = online_inference_pandas(
    url,
    df_validation_transformed
)

response_pandas.content

b'{"predictions": [0]}'

In [ ]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    ...

    response = requests.post()
    return response

In [21]:
# define a function to implement online inference with mlflow - json input

def online_inference_json(url: str, input: dict):

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        data=json.dumps(input)
    )

    return response

In [22]:
response_json = online_inference_json(
    url,
    {
        "dataframe_split": df_validation_transformed.to_dict(orient="split")
    }
)

response_json.content

b'{"predictions": [0]}'